# AI Professor — Indexação v2 (multilingual-e5-large + BM25 híbrido)
Melhorias sobre v1:
- Modelo: `intfloat/multilingual-e5-large` (1024d, melhor para português)
- Busca híbrida: dense + BM25 sparse com fusão RRF

In [ ]:
!pip install -q faster-whisper sentence-transformers 'qdrant-client[fastembed]'
!apt-get install -y -q ffmpeg

In [ ]:
from google.colab import userdata

QDRANT_URL = userdata.get('QDRANT_URL')
QDRANT_API_KEY = userdata.get('QDRANT_API_KEY')
COLLECTION = 'ai_professor_docs'
DENSE_MODEL = 'intfloat/multilingual-e5-large'
SPARSE_MODEL = 'Qdrant/bm25'
VECTOR_SIZE = 1024

In [ ]:
from google.colab import files
uploaded = files.upload()
video_path = list(uploaded.keys())[0]
print(f'Arquivo: {video_path}')

In [ ]:
import subprocess

audio_path = video_path.rsplit('.', 1)[0] + '.wav'
subprocess.run(
    ['ffmpeg', '-y', '-i', video_path,
     '-vn', '-acodec', 'pcm_s16le', '-ar', '16000', '-ac', '1', audio_path],
    check=True, capture_output=True
)
print(f'Áudio extraído: {audio_path}')

In [ ]:
from faster_whisper import WhisperModel

whisper = WhisperModel('large-v3', device='cuda', compute_type='float16')
segments_gen, info = whisper.transcribe(
    audio_path, language='pt', beam_size=5, vad_filter=True,
)
segments = list(segments_gen)
print(f'Duração: {info.duration:.0f}s | {len(segments)} segmentos')

In [ ]:
def chunk_segments(segments, max_words=400, overlap_words=50):
    chunks = []
    current_words = []
    current_start = 0.0
    for seg in segments:
        words = seg.text.strip().split()
        if not current_words:
            current_start = seg.start
        current_words.extend(words)
        if len(current_words) >= max_words:
            chunks.append({'text': ' '.join(current_words), 'start_sec': current_start, 'end_sec': seg.end})
            current_words = current_words[-overlap_words:]
            current_start = seg.start
    if current_words:
        chunks.append({'text': ' '.join(current_words), 'start_sec': current_start, 'end_sec': segments[-1].end if segments else 0.0})
    return chunks

chunks = chunk_segments(segments)
print(f'Total de chunks: {len(chunks)}')

In [ ]:
# Dense embeddings — multilingual-e5-large requer prefixo 'passage:' nos documentos
from sentence_transformers import SentenceTransformer

dense_model = SentenceTransformer(DENSE_MODEL)
texts = [c['text'] for c in chunks]
texts_passage = ['passage: ' + t for t in texts]
dense_vecs = dense_model.encode(texts_passage, batch_size=16, show_progress_bar=True, normalize_embeddings=True)
print(f'Dense shape: {dense_vecs.shape}')

In [ ]:
# Sparse embeddings — BM25 via fastembed
from fastembed.sparse.bm25 import Bm25

bm25 = Bm25(SPARSE_MODEL)
sparse_vecs = list(bm25.embed(texts))
print(f'Sparse gerado: {len(sparse_vecs)} vetores')

In [ ]:
# Criar collection com vetores nomeados: dense + sparse
import uuid
from qdrant_client import QdrantClient
from qdrant_client.http.models import (
    Distance, VectorParams, SparseVectorParams, SparseVector, PointStruct
)

client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)

if client.collection_exists(COLLECTION):
    client.delete_collection(COLLECTION)
    print(f'Collection "{COLLECTION}" removida.')

client.create_collection(
    collection_name=COLLECTION,
    vectors_config={'dense': VectorParams(size=VECTOR_SIZE, distance=Distance.COSINE)},
    sparse_vectors_config={'sparse': SparseVectorParams()},
)
print(f'Collection "{COLLECTION}" criada (dense={VECTOR_SIZE}d + sparse BM25).')

points = [
    PointStruct(
        id=str(uuid.uuid4()),
        vector={
            'dense': dense_vecs[i].tolist(),
            'sparse': SparseVector(
                indices=sparse_vecs[i].indices.tolist(),
                values=sparse_vecs[i].values.tolist(),
            ),
        },
        payload={
            'text': chunks[i]['text'],
            'source': video_path,
            'start_sec': chunks[i]['start_sec'],
            'end_sec': chunks[i]['end_sec'],
        },
    )
    for i in range(len(chunks))
]

client.upsert(collection_name=COLLECTION, points=points)
print(f'\n✅ {len(points)} chunks indexados.')

In [ ]:
# Verificação — busca híbrida com fusão RRF
from qdrant_client.http.models import Prefetch, Fusion

query = 'Como funciona o método AGG?'

# dense: prefixo 'query:'
q_dense = dense_model.encode('query: ' + query, normalize_embeddings=True).tolist()

# sparse: BM25
q_sparse = list(bm25.embed([query]))[0]
q_sparse_vec = SparseVector(indices=q_sparse.indices.tolist(), values=q_sparse.values.tolist())

results = client.query_points(
    collection_name=COLLECTION,
    prefetch=[
        Prefetch(query=q_dense, using='dense', limit=10),
        Prefetch(query=q_sparse_vec, using='sparse', limit=10),
    ],
    query=Fusion.RRF,
    limit=3,
).points

for r in results:
    print(f'Score: {r.score:.3f} | {r.payload["text"][:150]}...')
    print()